# Train TransUNet on Kaggle — 4-fold CV + fixed-test evaluation

Reproduces the local TransUNet experiment on a Kaggle GPU. TransUNet is
**image-only**, so it does **not** need the `data/bb_maps/` priors — you only
upload `data/splits`.

## Before you run
1. **Accelerator:** Settings → Accelerator → **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings → Internet → **On** (needed for `git clone` + `pip`).
3. **Data:** upload `data/splits.zip` (already built for you) as a **Kaggle Dataset**,
   then add it to this notebook (right panel → *Add Input*). To regenerate it,
   run `cd data && zip -r -0 splits.zip splits -x '*/yolox_coco/*'` (the exclude
   drops the YOLOX COCO symlinks, which TransUNet doesn't use and which otherwise
   double the archive). It must contain `splits/folds/fold_0..3/`, `splits/test/`,
   and `splits/class_map.txt`. Copy its mount path into `SPLITS_PATH` below.

Run the cells top to bottom. The full 4-fold run is ~2–5 h and fits one session.

## 1. Configure
Edit the values below to match your setup, then run.

In [ ]:
# === EDIT REPO_URL/REPO_DIR if needed. SPLITS_PATH is auto-detected in Step 3 ===
# === by finding class_map.txt under /kaggle/input — only set it manually if you ===
# === have several datasets attached and auto-detect picks the wrong one. ========
SPLITS_PATH = "/kaggle/input/pbl4-splits/splits"      # /kaggle/input/<dataset-slug>/splits
REPO_URL    = "https://github.com/Huay0804/PBL4.git"  # private? use https://<TOKEN>@github.com/Huay0804/PBL4.git
REPO_DIR    = "/kaggle/working/PBL4"

## 2. Get the code and verify the environment
Uses Kaggle's preinstalled TensorFlow 2.x + Keras 3 stack as-is (CUDA is already
wired up on the GPU image). We deliberately do **not** pip-upgrade
TensorFlow/numpy: scipy and scikit-image on Kaggle are built against the
preinstalled numpy 1.26 ABI, so upgrading numpy leaves them in a half-broken
state that crashes `import tensorflow` and can't be repaired in-session.

In [ ]:
import os, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

# Use Kaggle's preinstalled TF/Keras/numpy/scipy stack. Do NOT pip-install
# tensorflow or numpy here — that breaks scipy/scikit-image on Kaggle (they're
# compiled against the preinstalled numpy 1.26 ABI) and the resulting mismatch
# cannot be fixed by a kernel restart.
try:
    import tensorflow as tf
    import keras
except Exception as e:
    raise SystemExit(
        f"Environment broken (TF import failed): {e}\n\n"
        "Most likely a previous run pip-upgraded numpy/TF in this session and "
        "left a mixed install. Top-right → 'Stop session', then reopen this "
        "notebook to start a clean session with Kaggle's pristine stack."
    )

print(f"Preinstalled: TF {tf.__version__} | Keras {keras.__version__}")
assert keras.__version__.startswith("3."), (
    f"Need Keras 3.x; have {keras.__version__}. Stop session and start a fresh "
    "one — Kaggle's default GPU image ships Keras 3."
)
print("Environment OK — no pip install needed.")

## 3. Wire up data and sanity-check the environment
Links the read-only dataset to `data/splits` (the scripts resolve `data/splits/...`
relative to the repo root), then verifies TF/Keras/GPU and builds the model.

In [ ]:
import os, sys, glob, shutil
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))

# Resolve splits dir: use SPLITS_PATH if it has class_map.txt, otherwise
# auto-detect by globbing /kaggle/input (slug capitalization doesn't matter).
def _resolve_splits():
    if os.path.exists(os.path.join(SPLITS_PATH, "class_map.txt")):
        return SPLITS_PATH
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "No class_map.txt found anywhere under /kaggle/input — "
            "did you Add the dataset (right panel → Add Input)?"
        )
    if len(hits) > 1:
        print(f"Multiple candidates: {hits}")
        print(f"Using first: {hits[0]} — set SPLITS_PATH manually to override.")
    return os.path.dirname(hits[0])

splits_path = _resolve_splits()
print(f"Using SPLITS_PATH = {splits_path}")

# Link data/splits -> dataset. Replace any pre-existing entry safely
# (symlink, real directory, or file).
os.makedirs("data", exist_ok=True)
link = "data/splits"
if os.path.islink(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
elif os.path.exists(link):
    os.remove(link)
os.symlink(splits_path, link)
print(f"Linked data/splits -> {splits_path}")
print(f"Contents: {os.listdir('data/splits')}")

import tensorflow as tf, keras
print(f"\nTF {tf.__version__} | Keras {keras.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus or "NONE — confirm GPU accelerator is on (right panel → Settings)")

from segmentation_models import TransUNet
_m = TransUNet(input_shape=(512, 1024, 3), classes=33, activation="softmax")
print(f"TransUNet params: {_m.count_params():,}")
del _m

## 4. Train all 4 CV folds
Same command as local — the preset drives batch size, mixed precision,
early-stopping (patience 4) and the 5-epoch GPU-pool refresh loop.
`PBL4_GPU_DISPLAY_RESERVE_MB=0` uses the full headless GPU. If a session is cut
short, just re-run: folds merge incrementally into the CV summary.

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)
os.environ["PBL4_GPU_DISPLAY_RESERVE_MB"] = "0"

for k in range(4):
    print(f"\n================== TRAIN FOLD {k} ==================", flush=True)
    rc = subprocess.run(
        f"python -u scripts/train_segmentation_cv.py --model transunet --fold {k}",
        shell=True,
    ).returncode
    if rc != 0:
        raise SystemExit(f"Fold {k} training failed (exit {rc}).")
print("\nAll folds trained.")

## 5. Evaluate each fold on the fixed test set
Writes `test_summary.json`, `test_metrics.json`, `per_class_*`, `per_position_*`,
and `per_tooth_type_*` next to each fold's checkpoint (unchanged format).

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)

for k in range(4):
    print(f"\n================== EVAL FOLD {k} ==================", flush=True)
    rc = subprocess.run(
        f"python -u scripts/evaluate_final.py --model transunet --cv-fold {k}",
        shell=True,
    ).returncode
    if rc != 0:
        raise SystemExit(f"Fold {k} evaluation failed (exit {rc}).")
print("\nAll folds evaluated.")

## 6. Results
Cross-validation aggregate (validation folds) and the per-fold test-set summaries.

In [ ]:
import glob, json, os
os.chdir(REPO_DIR)

cv = "runs/cv/transunet_cv_summary.json"
if os.path.exists(cv):
    print("=== CV aggregate (validation) ===")
    print(json.dumps(json.load(open(cv)).get("aggregate", {}), indent=2))

print("\n=== Per-fold test-set summaries ===")
for p in sorted(glob.glob("runs/cv/fold_*/transunet/**/test_summary.json", recursive=True)):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2))

## 7. Download results
Zips the CV outputs (checkpoints + metrics) to the notebook **Output** tab,
excluding bulky TensorBoard logs and backup checkpoints. You can also
**Save Version** to persist `/kaggle/working` automatically.

In [ ]:
import os
os.chdir(REPO_DIR)
!zip -r -q /kaggle/working/transunet_runs.zip runs/cv -x "*/logs/*" "*/.training_backup/*"
print("Wrote /kaggle/working/transunet_runs.zip — download it from the Output tab.")